# Self Attention

In [1]:
import numpy as np
import math

In [2]:
L, d_k, d_v = 4, 8, 8

q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)


In [3]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)

Q
 [[ 0.5451241  -0.3796753  -0.24489294 -0.37896003  0.47657526 -0.53560703
   0.30233434 -1.95895616]
 [-1.55496153  0.31322333  0.23700928 -1.56396752  0.7658488   0.05075219
   0.19106612 -1.65792609]
 [ 0.60455251 -0.55350549 -0.26200273  2.26325615 -0.45431367  0.99133665
   0.17356109 -0.39098269]
 [-0.13946831 -0.05171317  1.01801396 -0.54152369  0.69097614 -0.7043838
  -0.58901648 -0.27440113]]
K
 [[ 1.13694082 -0.03424722  0.57358045  0.41457739  0.20446942 -0.07379709
  -1.72282358 -0.49293736]
 [-0.71348527 -1.43966766 -0.65041232 -0.23912386  0.08106064 -0.35374296
   0.06443396 -1.3502166 ]
 [-0.70823726 -1.17167633  1.01409108 -0.16510608 -0.16876955 -0.591565
   0.78280076 -0.51496947]
 [-0.2370395  -0.39618391  0.1136417  -0.35989185 -0.8483285   0.57650129
   0.60026087 -0.35167413]]
V
 [[ 1.41664252 -0.70463382 -1.37941546  0.86897613  0.29881586 -0.44700476
   0.47533093 -0.04617927]
 [ 0.91815832 -0.22057733  0.58749897 -0.16239539  1.81030356 -0.88448274
  -0.3628

$$
\text{Self Attention}(Q, K, V) =
\text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right) V
$$


In [4]:
np.matmul(q, k.T)

array([[ 0.91694785,  3.30016242,  1.35488937,  0.28708304],
       [-1.65014016,  3.17333027,  2.07692772,  0.91159314],
       [ 1.22197546,  0.14632512, -0.59156717,  0.43027979],
       [ 1.54591226,  0.27905086,  1.26143422, -0.88519046]])

In [5]:
# why we need sqrt(d_k) in denominator
q.var(), k.var(), np.matmul(q, k.T).var()

(np.float64(0.720468911796939),
 np.float64(0.45660222756349567),
 np.float64(1.6633449655285213))

In [7]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
scaled

array([[ 0.32419002,  1.16678361,  0.47902573,  0.10149918],
       [-0.58341265,  1.12194168,  0.73430484,  0.32229684],
       [ 0.43203357,  0.05173374, -0.20915058,  0.15212688],
       [ 0.54656252,  0.09865938,  0.44598435, -0.31296209]])

In [8]:
q.var(), k.var(), scaled.var()

(np.float64(0.720468911796939),
 np.float64(0.45660222756349567),
 np.float64(0.20791812069106513))

# Masking
- This is to ensure words don't get context from words generated in the future
- Not required in the encoders, but required in the decoders

In [24]:
mask = np.triu(np.ones((L, L), dtype=float), k=1)
mask

array([[0., 1., 1., 1.],
       [0., 0., 1., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 0.]])

In [25]:
mask[mask == 0] = 0.0
mask[mask == 1] = -np.inf

mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [26]:
scaled + mask

array([[ 0.32419002,        -inf,        -inf,        -inf],
       [-0.58341265,  1.12194168,        -inf,        -inf],
       [ 0.43203357,  0.05173374, -0.20915058,        -inf],
       [ 0.54656252,  0.09865938,  0.44598435, -0.31296209]])

In [27]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=1)).T

$$
\text{softmax}(x_i)
= \frac{\exp(x_i)}{\sum_{j=1}^{n} \exp(x_j)}
$$

In [30]:
attention = softmax(scaled + mask)
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.15376725, 0.84623275, 0.        , 0.        ],
       [0.4524222 , 0.30930133, 0.23827647, 0.        ],
       [0.3370812 , 0.21538361, 0.30482738, 0.14270781]])

In [31]:
new_v = np.matmul(attention, v)

new_v

array([[ 1.41664252, -0.70463382, -1.37941546,  0.86897613,  0.29881586,
        -0.44700476,  0.47533093, -0.04617927],
       [ 0.99480887, -0.29500936,  0.28505195, -0.00380422,  1.57788625,
        -0.81721295, -0.23396507, -0.32394453],
       [ 0.75474832, -0.35179558, -0.41843425,  0.34193133,  0.69880934,
        -0.92179555,  0.43852439,  0.13013999],
       [ 0.14646381, -0.33450474, -0.16556049,  0.35431679,  0.54349594,
        -1.02728798,  0.59321579,  0.05159952]])

In [32]:
v

array([[ 1.41664252, -0.70463382, -1.37941546,  0.86897613,  0.29881586,
        -0.44700476,  0.47533093, -0.04617927],
       [ 0.91815832, -0.22057733,  0.58749897, -0.16239539,  1.81030356,
        -0.88448274, -0.36284982, -0.3744167 ],
       [-0.71412753,  0.14781676,  0.10042831, -0.00412818,  0.0154825 ,
        -1.87172902,  1.40888459,  1.119876  ],
       [-2.18019169, -0.662443  ,  0.99688712,  0.68417396,  0.33734324,
        -0.80972257,  0.57233011, -1.35633759]])

# Function

In [36]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=1)).T

def scaled_do_product_attention(q, k, v, mask = None):
    d_k = q.shape[-1]
    scaled = np.matmul(q, k.T)/math.sqrt(d_k)
    if mask is not None:
        scaled  = scaled + mask
    
    attention = softmax(scaled)
    out = np.matmul(attention, v)

    return out, attention

In [38]:
values, attention = scaled_do_product_attention(q, k, v, mask = mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("New v\n", values)
print("Attention\n", attention)

Q
 [[ 0.5451241  -0.3796753  -0.24489294 -0.37896003  0.47657526 -0.53560703
   0.30233434 -1.95895616]
 [-1.55496153  0.31322333  0.23700928 -1.56396752  0.7658488   0.05075219
   0.19106612 -1.65792609]
 [ 0.60455251 -0.55350549 -0.26200273  2.26325615 -0.45431367  0.99133665
   0.17356109 -0.39098269]
 [-0.13946831 -0.05171317  1.01801396 -0.54152369  0.69097614 -0.7043838
  -0.58901648 -0.27440113]]
K
 [[ 1.13694082 -0.03424722  0.57358045  0.41457739  0.20446942 -0.07379709
  -1.72282358 -0.49293736]
 [-0.71348527 -1.43966766 -0.65041232 -0.23912386  0.08106064 -0.35374296
   0.06443396 -1.3502166 ]
 [-0.70823726 -1.17167633  1.01409108 -0.16510608 -0.16876955 -0.591565
   0.78280076 -0.51496947]
 [-0.2370395  -0.39618391  0.1136417  -0.35989185 -0.8483285   0.57650129
   0.60026087 -0.35167413]]
V
 [[ 1.41664252 -0.70463382 -1.37941546  0.86897613  0.29881586 -0.44700476
   0.47533093 -0.04617927]
 [ 0.91815832 -0.22057733  0.58749897 -0.16239539  1.81030356 -0.88448274
  -0.3628